In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('torchinfo') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torchinfo'])

from torchinfo import summary

  Using cached torchinfo-1.8.0-py3-none-any.whl.metadata (21 kB)
Using cached torchinfo-1.8.0-py3-none-any.whl (23 kB)
torchinfo siap digunakan pada kernel ini


In [1]:
input_size = (1, 3, 24, 24)

In [ ]:
import torch
import torch.nn as nn
from torchinfo import summary

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, padding=0), 
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

    def forward(self, x):
        x = self.block1(x)
        return x

model = SimpleCNN()

model_stats = summary(
    model, 
    input_size=input_size, 
    col_names=["input_size", "output_size", "num_params", "mult_adds"], 
    verbose=0
)
print(model_stats)

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Mult-Adds
SimpleCNN                                [1, 3, 24, 24]            [1, 16, 10, 10]           --                        --
├─Sequential: 1-1                        [1, 3, 24, 24]            [1, 16, 10, 10]           --                        --
│    └─Conv2d: 2-1                       [1, 3, 24, 24]            [1, 16, 20, 20]           1,216                     486,400
│    └─ReLU: 2-2                         [1, 16, 20, 20]           [1, 16, 20, 20]           --                        --
│    └─MaxPool2d: 2-3                    [1, 16, 20, 20]           [1, 16, 10, 10]           --                        --
Total params: 1,216
Trainable params: 1,216
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.49
Input size (MB): 0.01
Forward/backward pass size (MB): 0.05
Params size (MB): 0.00
Estimated Total Size (MB): 0.06


In [ ]:
import torch
import torch.nn as nn
from torchinfo import summary

class SimpleCNNDepthwise(nn.Module):
    def __init__(self):
        super(SimpleCNNDepthwise, self).__init__()

        # Block 1 dengan Depthwise Separable Conv
        self.block1 = nn.Sequential( 
            # Depthwise
            nn.Conv2d(in_channels=3, out_channels=3, kernel_size=5, padding=0, groups=3, bias=False),
            # Pointwise
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=1, bias=False),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        

    def forward(self, x):
        x = self.block1(x)
        return x

model_dw = SimpleCNNDepthwise()

model_dw_stats = summary(
    model_dw, 
    input_size=(1, 3, 24, 24), 
    col_names=["input_size", "output_size", "num_params", "mult_adds"], 
    verbose=0
)
print(model_dw_stats)

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Mult-Adds
SimpleCNNDepthwise                       [1, 3, 24, 24]            [1, 16, 10, 10]           --                        --
├─Sequential: 1-1                        [1, 3, 24, 24]            [1, 16, 10, 10]           --                        --
│    └─Conv2d: 2-1                       [1, 3, 24, 24]            [1, 3, 20, 20]            75                        30,000
│    └─Conv2d: 2-2                       [1, 3, 20, 20]            [1, 16, 20, 20]           48                        19,200
│    └─ReLU: 2-3                         [1, 16, 20, 20]           [1, 16, 20, 20]           --                        --
│    └─MaxPool2d: 2-4                    [1, 16, 20, 20]           [1, 16, 10, 10]           --                        --
Total params: 123
Trainable params: 123
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.05
Input size (MB): 0.01
Forwa

In [ ]:
# Comparison
params_standard = sum(p.numel() for p in model.parameters()) 

params_dw = sum(p.numel() for p in model_dw.parameters())
kehematan = ((params_standard - params_dw) / params_standard) * 100

print(f"\nCOMPARISON:")
print(f"- Standard SimpleCNN   : {params_standard} parameter")
print(f"- Depthwise SimpleCNN  : {params_dw} parameter")
print(f"- Efisiciency    : {kehematan:.2f}%")


COMPARISON:
- Standard SimpleCNN   : 14048 parameter
- Depthwise SimpleCNN  : 123 parameter
- Efisiciency    : 99.12%
